[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Perform OCR with GLM-OCR

---

[![arXiv](https://img.shields.io/badge/arXiv-2603.10910-b31b1b.svg)](https://arxiv.org/abs/2603.10910)

[GLM-OCR](https://huggingface.co/zai-org/GLM-OCR) is a 0.9B-parameter
vision-language model built for optical character recognition. It scores
94.62 on OmniDocBench V1.5, placing it first on that benchmark. The
architecture pairs a CogViT visual encoder with a GLM-0.5B language
decoder through a cross-modal connector that downsamples visual tokens.

The model handles documents at resolutions up to 8K (7680x4320) in 8+
languages. Three built-in modes cover text recognition, LaTeX formula
recognition, and table recognition. Custom prompts can extract
structured data like JSON from documents.

We test GLM-OCR on eight datasets below: captchas, LaTeX equations,
receipts, date stamps, jersey numbers, container serials, tire codes,
and license plates.

## Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi`
command to do that. In case of any problems navigate to `Edit` →
`Notebook settings` → `Hardware accelerator`, set it to `L4 GPU`,
and then click `Save`.

In [1]:
!nvidia-smi

Thu Apr  2 15:18:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    gpu_name = torch.cuda.get_device_name()
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE = torch.float16
    gpu_name = "Apple Silicon (MPS)"
else:
    DEVICE = torch.device("cpu")
    DTYPE = torch.float32
    gpu_name = "CPU"

print(f"Device : {DEVICE} ({gpu_name})")
print(f"Dtype  : {DTYPE}")

Device : cuda (Tesla T4)
Dtype  : torch.bfloat16


### Configure your API keys

To run this notebook you need a HuggingFace Token (to download the
model) and a Roboflow API Key (to download the evaluation datasets).
Follow these steps:

- Open your [`HuggingFace Settings`](https://huggingface.co/settings) page. Click `Access Tokens` then `New Token` to generate new token.
- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on `Secrets` (🔑).
    - Store HuggingFace Access Token under the name `HF_TOKEN`.
    - Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

In [3]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

SecretNotFoundError: Secret HF_TOKEN does not exist.

## Install dependencies

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git roboflow

## Load GLM-OCR model

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "zai-org/GLM-OCR"

processor = AutoProcessor.from_pretrained(MODEL_ID)

if DEVICE.type == "cuda":
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
        device_map="auto",
    )
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
    ).to(DEVICE)

## Define helper functions

In [ ]:
import json
import os
import textwrap
import random
from glob import glob

from tqdm import tqdm

import matplotlib.pyplot as plt
from PIL import Image

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def run_ocr(image: Image.Image, prompt: str = "Text Recognition:") -> str:
    """Run GLM-OCR on a single PIL image and return the decoded text."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    inputs.pop("token_type_ids", None)
    generated_ids = model.generate(**inputs, max_new_tokens=8192)
    return processor.decode(
        generated_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()


def load_images(
    dataset_location: str,
    split: str = "test",
) -> list[dict]:
    """Load image paths from a Roboflow dataset directory.

    Tries the requested split first, then falls back to valid, then train.
    Returns a list of dicts with an 'image_path' key.
    """
    for candidate in [split, "valid", "train"]:
        split_dir = os.path.join(dataset_location, candidate)
        if os.path.isdir(split_dir):
            paths = sorted([
                os.path.join(split_dir, f)
                for f in os.listdir(split_dir)
                if f.lower().endswith(IMAGE_EXTENSIONS)
            ])
            if paths:
                random.shuffle(paths)
                print(f"Loaded {len(paths)} images from {candidate} split")
                return [{"image_path": p} for p in paths]
    print(f"No images found in {dataset_location}")
    return []


def run_and_plot(
    entries: list[dict],
    prompt: str,
    n: int = 9,
    wrap_width: int = 40,
):
    """Run OCR on up to n entries and display results in a 3-column grid."""
    entries = entries[:n]
    if not entries:
        print("No entries to display.")
        return

    cols = 3
    rows = (len(entries) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else list(axes)
    else:
        axes = axes.flatten()

    for idx, entry in enumerate(tqdm(entries, desc="Running OCR")):
        if "image" in entry and isinstance(entry["image"], Image.Image):
            image = entry["image"]
        else:
            image = Image.open(entry["image_path"]).convert("RGB")

        prediction = run_ocr(image, prompt)

        axes[idx].imshow(image)
        axes[idx].axis("off")
        pred_wrapped = textwrap.fill(prediction, width=wrap_width).replace("$", "\\$")
        axes[idx].set_title(f"Pred: {pred_wrapped}", fontsize=8, loc="left", pad=8)

    for idx in range(len(entries), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()


## 1. Captcha text recognition

Distorted captcha images containing 4–6 alphanumeric characters with
noise, warping, and color variations.

<a href="https://universe.roboflow.com/roboflow-jvuqo/captcha-ocr-g5ol8">
    <img src="https://app.roboflow.com/images/download-dataset-badge.svg">
</a>

In [ ]:
from roboflow import download_dataset

captcha_dataset = download_dataset(
    "https://universe.roboflow.com/roboflow-jvuqo/captcha-ocr-g5ol8/1",
    "jsonl",
)

captcha_entries = load_images(captcha_dataset.location)

In [ ]:
run_and_plot(captcha_entries, prompt="Text Recognition:")

## 2. LaTeX equation recognition

Images of mathematical equations. The model is prompted to return
the LaTeX source.

<a href="https://universe.roboflow.com/roboflow-jvuqo/latex-ocr-subset">
    <img src="https://app.roboflow.com/images/download-dataset-badge.svg">
</a>

In [ ]:
latex_dataset = download_dataset(
    "https://universe.roboflow.com/roboflow-jvuqo/latex-ocr-subset/3",
    "jsonl",
)

latex_entries = load_images(latex_dataset.location)

In [ ]:
run_and_plot(latex_entries, prompt="Formula Recognition:")

## 3. Receipt parsing (structured JSON)

Indonesian receipt images.

<a href="https://universe.roboflow.com/roboflow-jvuqo/cord-v2-receipts">
    <img src="https://app.roboflow.com/images/download-dataset-badge.svg">
</a>

In [ ]:
receipt_dataset = download_dataset(
    "https://universe.roboflow.com/roboflow-jvuqo/cord-v2-receipts/1",
    "jsonl",
)

receipt_entries = load_images(receipt_dataset.location)

In [ ]:
run_and_plot(receipt_entries, prompt="What is the total price on this receipt?")

## 4. Date stamp extraction

Images containing date stamps. The model extracts the date in a
consistent format.

<a href="https://universe.roboflow.com/roboflow-jvuqo/date-stamp-extraction">
    <img src="https://app.roboflow.com/images/download-dataset-badge.svg">
</a>

In [ ]:
date_dataset = download_dataset(
    "https://universe.roboflow.com/roboflow-jvuqo/date-stamp-extraction/1",
    "jsonl",
)

date_entries = load_images(date_dataset.location)

In [ ]:
DATE_PROMPT = "Extract the date from this image. Return it in YYYY-MM-DD format."

run_and_plot(date_entries, prompt=DATE_PROMPT)

## 5. Basketball jersey number recognition

Images of basketball players. The model extracts the jersey number.

<a href="https://universe.roboflow.com/roboflow-jvuqo/basketball-jersey-numbers-ocr">
    <img src="https://app.roboflow.com/images/download-dataset-badge.svg">
</a>

In [ ]:
jersey_dataset = download_dataset(
    "https://universe.roboflow.com/roboflow-jvuqo/basketball-jersey-numbers-ocr/7",
    "jsonl",
)

jersey_entries = load_images(jersey_dataset.location)

In [ ]:
JERSEY_PROMPT = "Read jersey number."

run_and_plot(jersey_entries, prompt=JERSEY_PROMPT)

## 6. Container serial number recognition

Shipping container images with serial number regions.

<a href="https://universe.roboflow.com/kerofox/container-id-number">
    <img src="https://app.roboflow.com/images/download-dataset-badge.svg">
</a>

In [ ]:
container_dataset = download_dataset(
    "https://universe.roboflow.com/kerofox/container-id-number/dataset/3",
    "coco",
)

container_entries = load_images(container_dataset.location)

In [ ]:
run_and_plot(container_entries, prompt="Read only the container serial number.")

## 7. Tire code recognition

Tire images with embossed codes and digits.

<a href="https://universe.roboflow.com/bence-toth-h5oec/tire-serial">
    <img src="https://app.roboflow.com/images/download-dataset-badge.svg">
</a>

In [ ]:
tire_dataset = download_dataset(
    "https://universe.roboflow.com/bence-toth-h5oec/tire-serial/dataset/1",
    "coco",
)

tire_entries = load_images(tire_dataset.location)

In [ ]:
run_and_plot(tire_entries, prompt="Read tire serial number.")

## 8. License plate recognition

Vehicle images with license plates.

<a href="https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e">
    <img src="https://app.roboflow.com/images/download-dataset-badge.svg">
</a>

In [ ]:
plate_dataset = download_dataset(
    "https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e/dataset/1",
    "coco",
)

plate_entries = load_images(plate_dataset.location)

In [ ]:
run_and_plot(plate_entries, prompt="Read license plate number.")